# Process Recordings Pipeline

Dataset: `process_recordings.csv`

This notebook is organized into one cell per analytics phase (Ch. 1-17) and includes both explanatory (causal-oriented) and predictive modeling workflows.

Artifacts are written to `ml-pipelines/artifacts/`.

## 1) Problem Framing (Ch. 1)

**Business question:**
How can we use `process_recordings.csv` to improve service operations and outcomes by forecasting key targets while also understanding the drivers behind those outcomes?

**Modeling goals (kept distinct):**
- **Predictive model:** maximize out-of-sample performance on unseen records.
- **Explanatory model:** estimate and interpret relationships between important predictors and the outcome for decision support.

**Success metrics:**
- Predictive: MAE/RMSE (regression) or F1/AUC (classification), plus stability across folds.
- Explanatory: coefficient sign/direction consistency, statistical significance, and practical effect size.

**Decision requirement:**
For this pipeline, produce **both** models and explicitly report:
1) most impactful features, and
2) recommended operational decisions based on those effects.

## 2) Data Acquisition and Preparation (Ch. 2-5, 7)

- Load `process_recordings.csv` (and any linked tables if needed).
- Standardize column names and data types.
- Handle missing values, duplicates, invalid timestamps, and outliers.
- Engineer reusable features (time-based, lag features, rates, interaction terms).
- Encode categorical variables and scale numeric inputs where appropriate.
- Build a reproducible preprocessing pipeline (e.g., `ColumnTransformer` + `Pipeline`) rather than one-off transforms.

**Deliverable:** a versioned, repeatable preparation workflow that can be reused at training and inference time.

## 3) Exploration (Ch. 6, 8)

- Profile the target variable and key predictors.
- Visualize distributions and detect skew, long tails, and anomalies.
- Examine pairwise relationships and multicollinearity.
- Investigate subgroup differences (e.g., by location, staff type, process category, time windows).
- Document notable built-in patterns discovered in this dataset.

**Goal:** build intuition before modeling so feature choices and assumptions are evidence-driven.

## 4) Modeling (Ch. 9-14)

Train at least two model tracks:

1. **Explanatory (causal-oriented) model**
   - Prefer interpretable specification (e.g., OLS / regularized linear model with careful controls).
   - Focus on direction, magnitude, and confidence of effects.

2. **Predictive model**
   - Compare algorithms (e.g., linear baseline, tree-based, ensemble).
   - Optimize for predictive performance on held-out data.

**Requirement:** keep explanatory and predictive objectives separate; do not claim causality from a pure prediction model.

## 5) Evaluation and Selection (Ch. 15)

- Use train/test split and/or cross-validation.
- Tune hyperparameters for predictive candidates.
- Compare models with business-relevant metrics.
- Run fairness checks across relevant groups where applicable.
- Translate results into operational meaning (not only statistical scores).

**Selection rule:**
- Choose one best predictive model for deployment.
- Keep one explanatory model for stakeholder interpretation and policy guidance.

## 6) Feature Selection (Ch. 16)

- Rank candidate predictors using model-based importance, permutation importance, or coefficient-based evidence.
- Remove weak/redundant features while checking performance impact.
- Validate that selected features are stable across folds/samples.
- Document why top features matter from both domain and statistical perspectives.

**Required output in this notebook:**
1) top impactful features table,
2) brief explanation of why they matter,
3) concrete decisions the organization can make from those insights.

## 7) Deployment (Ch. 17)

- Serialize preprocessing pipeline + selected predictive model.
- Export schema, metrics, and top-feature artifacts to `ml-pipelines/artifacts/`.
- Expose predictions for application use (API, dashboard, or form-driven UI).
- Include explanatory outputs (feature effects and recommendations) in stakeholder-facing reporting.

**Deployment objective:** reliable predictions in production, paired with interpretable guidance for action.

In [1]:
# Setup and shared configuration
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
import joblib

RANDOM_STATE = 42

# Robust paths (works whether run from project root or ml-pipelines/)
candidate_artifact_dirs = [Path('ml-pipelines/artifacts'), Path('artifacts')]
ARTIFACT_DIR = next((p for p in candidate_artifact_dirs if p.parent.exists()), candidate_artifact_dirs[0])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

candidate_data_paths = [
    Path('datasets/process_recordings.csv'),
    Path('../datasets/process_recordings.csv'),
]
DATA_PATH = next((p for p in candidate_data_paths if p.exists()), candidate_data_paths[0])
assert DATA_PATH.exists(), f"Missing dataset. Checked: {[str(p) for p in candidate_data_paths]}"

print('Data path:', DATA_PATH)
print('Artifact path:', ARTIFACT_DIR)

Data path: ..\datasets\process_recordings.csv
Artifact path: ml-pipelines\artifacts


In [2]:
# Phase 2: Data Acquisition and Preparation (reproducible)
raw_df = pd.read_csv(DATA_PATH)


def pick_binary_target(df: pd.DataFrame) -> str:
    candidates = ['referral_made', 'concerns_flagged', 'progress_noted']
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f'No expected binary target found. Available columns: {df.columns.tolist()}')


def prepare_process_recordings(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    out = df.copy()

    # Standardize date + booleans
    out['session_date'] = pd.to_datetime(out.get('session_date'), errors='coerce')
    for c in ['progress_noted', 'concerns_flagged', 'referral_made', 'notes_restricted']:
        if c in out.columns:
            out[c] = out[c].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

    # Reusable engineered features
    out['session_month'] = out['session_date'].dt.month.fillna(0).astype(int)
    out['session_weekday'] = out['session_date'].dt.day_name().fillna('Unknown')
    out['intervention_count'] = out.get('interventions_applied', '').fillna('').astype(str).str.split(',').apply(
        lambda x: len([i for i in x if str(i).strip()])
    )
    out['narrative_length'] = out.get('session_narrative', '').fillna('').astype(str).str.len()

    # Keep stable modeling features
    keep_if_exists = [
        'session_type', 'session_duration_minutes', 'emotional_state_observed', 'emotional_state_end',
        'social_worker', 'follow_up_actions', 'notes_restricted',
        'intervention_count', 'narrative_length', 'session_month', 'session_weekday', target_col
    ]
    keep_cols = [c for c in keep_if_exists if c in out.columns]
    model_df = out[keep_cols].copy()

    # Basic quality cleanup
    if 'session_duration_minutes' in model_df.columns:
        model_df.loc[model_df['session_duration_minutes'] < 0, 'session_duration_minutes'] = np.nan

    return model_df


TARGET_COL = pick_binary_target(raw_df)
process_df = prepare_process_recordings(raw_df, TARGET_COL)
prepared_path = ARTIFACT_DIR / 'process_recordings_prepared.csv'
process_df.to_csv(prepared_path, index=False)

print('Selected target:', TARGET_COL)
print('Prepared shape:', process_df.shape)
process_df.head()

Selected target: referral_made
Prepared shape: (2819, 12)


,session_type,session_duration_minutes,emotional_state_observed,emotional_state_end,social_worker,follow_up_actions,notes_restricted,intervention_count,narrative_length,session_month,session_weekday,referral_made
0,Individual,62.0,Angry,Hopeful,SW-02,Referral to specialist,0,1,62,11,Wednesday,0
1,Group,83.0,Distressed,Sad,SW-10,Referral to specialist,0,1,57,11,Saturday,1
2,Individual,77.0,Anxious,Hopeful,SW-01,Referral to specialist,0,2,62,11,Friday,0
3,Group,77.0,Hopeful,Happy,SW-19,Schedule follow-up session,0,2,57,12,Friday,0
4,Individual,77.0,Happy,Happy,SW-15,Schedule follow-up session,0,1,62,12,Wednesday,0


In [3]:
# Phase 3: Exploration
print('Shape:', process_df.shape)
print('\nMissing values:')
print(process_df.isna().sum())

print(f'\nTarget distribution ({TARGET_COL}):')
print(process_df[TARGET_COL].value_counts(dropna=False))

for c in ['session_type', 'emotional_state_observed', 'emotional_state_end', 'follow_up_actions']:
    if c in process_df.columns:
        print(f'\n{c} distribution:')
        print(process_df[c].value_counts().head(10))

if {'session_duration_minutes', TARGET_COL}.issubset(process_df.columns):
    print('\nDuration summary by target:')
    print(process_df.groupby(TARGET_COL)['session_duration_minutes'].describe())

Shape: (2819, 12)

Missing values:
session_type                0
session_duration_minutes    0
emotional_state_observed    0
emotional_state_end         0
social_worker               0
follow_up_actions           0
notes_restricted            0
intervention_count          0
narrative_length            0
session_month               0
session_weekday             0
referral_made               0
dtype: int64

Target distribution (referral_made):
referral_made
0    2407
1     412
Name: count, dtype: int64

session_type distribution:
session_type
Individual    1805
Group         1014
Name: count, dtype: int64

emotional_state_observed distribution:
emotional_state_observed
Sad           499
Calm          476
Anxious       462
Angry         392
Hopeful       391
Withdrawn     356
Happy         150
Distressed     93
Name: count, dtype: int64

emotional_state_end distribution:
emotional_state_end
Hopeful      1178
Calm          896
Happy         435
Sad           154
Anxious       137
Withdrawn

In [4]:
# Phase 4: Modeling (explanatory + predictive)
X = process_df.drop(columns=[TARGET_COL])
y = process_df[TARGET_COL].astype(int)

categorical_features = [c for c in X.columns if X[c].dtype == 'object']
numeric_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
    ]
)

# Explanatory track: interpretable logistic regression
explanatory_model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=600, class_weight='balanced', random_state=RANDOM_STATE)),
])

# Predictive track: non-linear random forest
predictive_model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
    )),
])

stratify_y = y if y.nunique() > 1 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=stratify_y,
)

print('Train size:', X_train.shape, '| Test size:', X_test.shape)
print('Class balance (train):')
print(y_train.value_counts(normalize=True))

Train size: (1973, 11) | Test size: (846, 11)
Class balance (train):
referral_made
0    0.854029
1    0.145971
Name: proportion, dtype: float64


In [5]:
# Phase 5: Evaluation and Selection (CV + holdout + fairness)
minority_count = int(y_train.value_counts().min()) if y_train.nunique() > 1 else 0
n_splits = max(2, min(5, minority_count)) if minority_count >= 2 else 2

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc': 'roc_auc', 'f1': 'f1', 'recall': 'recall', 'precision': 'precision'}

cv_explanatory = cross_validate(explanatory_model, X_train, y_train, cv=cv, scoring=scoring)
cv_predictive = cross_validate(predictive_model, X_train, y_train, cv=cv, scoring=scoring)

print('Explanatory CV means:')
for metric in scoring:
    print(f"  {metric}: {cv_explanatory['test_' + metric].mean():.3f}")

print('\nPredictive CV means:')
for metric in scoring:
    print(f"  {metric}: {cv_predictive['test_' + metric].mean():.3f}")

explanatory_model.fit(X_train, y_train)
predictive_model.fit(X_train, y_train)

exp_proba = explanatory_model.predict_proba(X_test)[:, 1]
pred_proba = predictive_model.predict_proba(X_test)[:, 1]

exp_auc = roc_auc_score(y_test, exp_proba) if y_test.nunique() > 1 else np.nan
pred_auc = roc_auc_score(y_test, pred_proba) if y_test.nunique() > 1 else np.nan

best_model_name = 'predictive_model' if (np.isnan(exp_auc) or pred_auc >= exp_auc) else 'explanatory_model'
best_model = predictive_model if best_model_name == 'predictive_model' else explanatory_model

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f'\nHoldout ROC-AUC explanatory: {exp_auc:.3f}' if not np.isnan(exp_auc) else '\nHoldout ROC-AUC explanatory: NA')
print(f'Holdout ROC-AUC predictive:  {pred_auc:.3f}' if not np.isnan(pred_auc) else 'Holdout ROC-AUC predictive: NA')
print(f'\nSelected production model: {best_model_name}')

print('\nClassification report (selected model):')
print(classification_report(y_test, y_pred, digits=3, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

fairness_rows = []
y_proba_series = pd.Series(y_proba, index=X_test.index)
for group_col in [c for c in ['session_type', 'social_worker'] if c in X_test.columns]:
    grouped = X_test.copy()
    grouped['y_true'] = y_test
    grouped['y_pred'] = y_pred
    for g, sub in grouped.groupby(group_col):
        if sub['y_true'].nunique() < 2:
            group_auc = np.nan
        else:
            group_auc = roc_auc_score(sub['y_true'], y_proba_series.loc[sub.index])
        fairness_rows.append({
            'group_column': group_col,
            'group_value': g,
            'n': len(sub),
            'positive_rate_pred': sub['y_pred'].mean(),
            'recall': recall_score(sub['y_true'], sub['y_pred'], zero_division=0),
            'precision': precision_score(sub['y_true'], sub['y_pred'], zero_division=0),
            'auc': group_auc,
        })

fairness_df = pd.DataFrame(fairness_rows)
fairness_df.head(20)

Explanatory CV means:
  roc_auc: 0.458
  f1: 0.182
  recall: 0.368
  precision: 0.121

Predictive CV means:
  roc_auc: 0.440
  f1: 0.038
  recall: 0.028
  precision: 0.066

Holdout ROC-AUC explanatory: 0.478
Holdout ROC-AUC predictive:  0.490

Selected production model: predictive_model

Classification report (selected model):
              precision    recall  f1-score   support

           0      0.850     0.936     0.891       722
           1      0.098     0.040     0.057       124

    accuracy                          0.805       846
   macro avg      0.474     0.488     0.474       846
weighted avg      0.740     0.805     0.769       846

Confusion matrix:
[[676  46]
 [119   5]]


,group_column,group_value,n,positive_rate_pred,recall,precision,auc
0,session_type,Group,322,0.021739,0.021277,0.142857,0.467930
1,session_type,Individual,524,0.083969,0.051948,0.090909,0.499898
2,social_worker,SW-01,42,0.000000,0.000000,0.000000,0.506122
3,social_worker,SW-02,40,0.000000,0.000000,0.000000,0.496094
4,social_worker,SW-03,38,0.026316,0.000000,0.000000,0.602941
5,social_worker,SW-04,41,0.048780,0.000000,0.000000,0.382353
6,social_worker,SW-05,47,0.297872,0.142857,0.071429,0.367857
7,social_worker,SW-06,41,0.073171,0.125000,0.333333,0.492424
8,social_worker,SW-07,37,0.000000,0.000000,0.000000,0.620690
9,social_worker,SW-08,35,0.028571,0.000000,0.000000,0.604000


In [6]:
# Phase 6: Feature Selection + business recommendations
# Explanatory coefficients
exp_clf = explanatory_model.named_steps['clf']
exp_feature_names = explanatory_model.named_steps['prep'].get_feature_names_out()
exp_coeff = pd.DataFrame({
    'feature': exp_feature_names,
    'coefficient': exp_clf.coef_[0],
    'abs_coefficient': np.abs(exp_clf.coef_[0]),
}).sort_values('abs_coefficient', ascending=False)

# Predictive importances
pred_clf = predictive_model.named_steps['clf']
pred_feature_names = predictive_model.named_steps['prep'].get_feature_names_out()
pred_importance = pd.DataFrame({
    'feature': pred_feature_names,
    'importance': pred_clf.feature_importances_,
}).sort_values('importance', ascending=False)

top_features = pred_importance.head(15).copy()

# Export feature artifacts
top_features_path = ARTIFACT_DIR / 'process_recordings_top_features.csv'
top_features.to_csv(top_features_path, index=False)

exp_coeff_path = ARTIFACT_DIR / 'process_recordings_top_explanatory_coefficients.csv'
exp_coeff.head(25).to_csv(exp_coeff_path, index=False)

print('Top predictive features:')
display(top_features)

print('\nTop explanatory coefficients (absolute magnitude):')
display(exp_coeff.head(15))

# Data-driven recommendation notes
recommendations = [
    'Prioritize additional support workflows for sessions that align with high-risk feature patterns.',
    'Use top predictive features to trigger proactive referrals earlier in the case lifecycle.',
    'Use explanatory coefficients to brief leadership on which session characteristics are most associated with target outcomes.',
    'Monitor fairness metrics by session_type and social_worker; retrain if large disparities persist.',
]

recommendation_path = ARTIFACT_DIR / 'process_recordings_recommendations.txt'
recommendation_path.write_text('\n'.join(f'- {r}' for r in recommendations), encoding='utf-8')

print('\nRecommendations path:', recommendation_path)

Top predictive features:


,feature,importance
48,num__session_duration_minutes,0.148428
52,num__session_month,0.091311
50,num__intervention_count,0.045102
51,num__narrative_length,0.023962
3,cat__emotional_state_observed_Anxious,0.023092
40,cat__follow_up_actions_Schedule follow-up session,0.022365
13,cat__emotional_state_end_Hopeful,0.021316
20,cat__social_worker_SW-05,0.021245
11,cat__emotional_state_end_Calm,0.020831
37,cat__follow_up_actions_Coordinate with family,0.020678



Top explanatory coefficients (absolute magnitude):


,feature,coefficient,abs_coefficient
20,cat__social_worker_SW-05,0.644555,0.644555
31,cat__social_worker_SW-16,-0.635941,0.635941
34,cat__social_worker_SW-19,0.436493,0.436493
24,cat__social_worker_SW-09,0.403253,0.403253
3,cat__emotional_state_observed_Anxious,-0.357875,0.357875
16,cat__social_worker_SW-01,-0.345525,0.345525
26,cat__social_worker_SW-11,-0.342317,0.342317
27,cat__social_worker_SW-12,0.318704,0.318704
0,cat__session_type_Group,-0.313683,0.313683
7,cat__emotional_state_observed_Hopeful,-0.311146,0.311146



Recommendations path: ml-pipelines\artifacts\process_recordings_recommendations.txt


In [7]:
# Phase 7: Deployment artifacts
model_path = ARTIFACT_DIR / 'process_recordings_predictive_model.joblib'
joblib.dump(best_model, model_path)

metrics_holdout = {
    'roc_auc': float(roc_auc_score(y_test, y_proba)) if y_test.nunique() > 1 else None,
    'f1': float(f1_score(y_test, y_pred, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred, zero_division=0)),
    'precision': float(precision_score(y_test, y_pred, zero_division=0)),
}

schema = {
    'dataset': 'process_recordings.csv',
    'target': TARGET_COL,
    'selected_model': best_model_name,
    'features': X.columns.tolist(),
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'metrics_holdout': metrics_holdout,
}

schema_path = ARTIFACT_DIR / 'process_recordings_model_schema.json'
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2)

metrics_path = ARTIFACT_DIR / 'process_recordings_model_metrics.csv'
pd.DataFrame([metrics_holdout]).to_csv(metrics_path, index=False)

fairness_path = ARTIFACT_DIR / 'process_recordings_fairness_report.csv'
fairness_df.to_csv(fairness_path, index=False)

print('Artifacts written:')
print('-', model_path)
print('-', schema_path)
print('-', metrics_path)
print('-', fairness_path)
print('-', top_features_path)
print('-', exp_coeff_path)
print('-', recommendation_path)

Artifacts written:
- ml-pipelines\artifacts\process_recordings_predictive_model.joblib
- ml-pipelines\artifacts\process_recordings_model_schema.json
- ml-pipelines\artifacts\process_recordings_model_metrics.csv
- ml-pipelines\artifacts\process_recordings_fairness_report.csv
- ml-pipelines\artifacts\process_recordings_top_features.csv
- ml-pipelines\artifacts\process_recordings_top_explanatory_coefficients.csv
- ml-pipelines\artifacts\process_recordings_recommendations.txt


## Interpretation for Stakeholders

- **Predictive pipeline use:** score new session records and flag likely cases requiring referral/action.
- **Explanatory pipeline use:** use coefficient direction and magnitude to explain which session factors align with higher/lower outcome likelihood.
- **Most impactful features:** review `process_recordings_top_features.csv`.
- **Decision recommendations:** review `process_recordings_recommendations.txt` for operational actions.
- **Governance:** monitor fairness slices and retrain as process patterns shift over time.